# Bank Statement Transaction Extraction with Llama Vision

Specialized notebook for extracting transactional data from bank statements using Llama-3.2-Vision-Instruct.

**Key Features:**
- Handles empty debit/credit cells correctly
- Preserves row-by-row transaction structure
- Calculates running balances
- Validates transaction integrity

In [ ]:
# Configuration
from pathlib import Path
MODEL_PATH = "/home/jovyan/nfs_share/models/Llama-3.2-11B-Vision-Instruct"

# Dynamically discover all PNG images in evaluation_data directory
EVALUATION_DATA_DIR = Path("evaluation_data")

# Automatically find all PNG files in the evaluation_data directory
BANK_STATEMENT_IMAGES = {}
if EVALUATION_DATA_DIR.exists():
    for png_file in sorted(EVALUATION_DATA_DIR.glob("*.png")):
        # Use the filename (without extension) as the key
        key = png_file.stem.upper()
        BANK_STATEMENT_IMAGES[key] = str(png_file)

# Ground truth CSV for evaluation
GROUND_TRUTH_CSV = "evaluation_data/ground_truth.csv"

# Model loading settings - NO QUANTIZATION BY DEFAULT (H200 has plenty of memory)
USE_QUANTIZATION = False  # H200 doesn't need quantization
LOAD_IN_8BIT = False  # 8-bit causes inference issues with bitsandbytes  
DEVICE_MAP = "auto"  # Automatic device mapping
MAX_NEW_TOKENS = 6000  # INCREASED - was 3000, now 6000 for complete extraction

# Rich imports for pretty printing
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.text import Text
from rich.progress import track
from rich import print as rprint
from rich.syntax import Syntax

# Initialize Rich console
console = Console()
rprint("[bold blue]🚀 Configuration loaded with Rich formatting enabled[/bold blue]")

# Display discovered images in a nice table
if BANK_STATEMENT_IMAGES:
    rprint(f"[green]✅ Found {len(BANK_STATEMENT_IMAGES)} bank statement images[/green]")
    
    image_table = Table(title="📄 Discovered Bank Statement Images", border_style="cyan")
    image_table.add_column("Key", style="yellow", no_wrap=True)
    image_table.add_column("File Path", style="green")
    
    for key, path in BANK_STATEMENT_IMAGES.items():
        image_table.add_row(key, path)
    
    console.print(image_table)
else:
    rprint("[yellow]⚠️ No PNG images found in evaluation_data directory[/yellow]")
    rprint("[yellow]Please ensure PNG files exist in the evaluation_data/ directory[/yellow]")

In [ ]:
import torch
from transformers import MllamaForConditionalGeneration, AutoProcessor
from PIL import Image
import json
import re
from typing import Dict, List, Any, Optional, Tuple
import pandas as pd
from datetime import datetime
from decimal import Decimal, InvalidOperation
import warnings

warnings.filterwarnings('ignore')

rprint("[bold green]✅ Core libraries imported successfully[/bold green]")

In [ ]:
# Select an image for testing
# You can change this to any key from BANK_STATEMENT_IMAGES
if BANK_STATEMENT_IMAGES:
    # Use IMAGE_003 if available (equivalent to old CBA_BASIC), otherwise use first available image
    if "IMAGE_003" in BANK_STATEMENT_IMAGES:
        STATEMENT_IMAGE_PATH = BANK_STATEMENT_IMAGES["IMAGE_003"]
        rprint("[cyan]📌 Using IMAGE_003 (default test image)[/cyan]")
    else:
        # Use the first available image
        first_key = list(BANK_STATEMENT_IMAGES.keys())[0]
        STATEMENT_IMAGE_PATH = BANK_STATEMENT_IMAGES[first_key]
        rprint(f"[cyan]📌 Using {first_key} (first available image)[/cyan]")
else:
    rprint("[red]❌ No images available to test![/red]")
    STATEMENT_IMAGE_PATH = None

# Load and display the bank statement image with Rich formatting
from IPython.display import display

if STATEMENT_IMAGE_PATH:
    rprint(f"[bold cyan]📄 Loading bank statement image:[/bold cyan] [yellow]{STATEMENT_IMAGE_PATH}[/yellow]")

    try:
        image = Image.open(STATEMENT_IMAGE_PATH)
        
        # Create info table
        image_table = Table(title="🖼️ Image Information", border_style="blue")
        image_table.add_column("Property", style="cyan", no_wrap=True)
        image_table.add_column("Value", style="green")
        
        image_table.add_row("File Path", STATEMENT_IMAGE_PATH)
        image_table.add_row("Dimensions", f"{image.size[0]} × {image.size[1]} pixels")
        image_table.add_row("Format", image.format or "Unknown")
        image_table.add_row("Mode", image.mode)
        
        console.print(image_table)
        rprint("[bold green]✅ Image loaded successfully![/bold green]")
        
        # Display the image in the notebook
        rprint("[dim]Displaying image below...[/dim]")
        display(image)
        
    except Exception as e:
        rprint(f"[bold red]❌ Error loading image:[/bold red] {e}")
        rprint(f"[yellow]💡 Check that the file exists at:[/yellow] [cyan]{STATEMENT_IMAGE_PATH}[/cyan]")

In [ ]:
# Initialize model and processor
rprint("[bold yellow]Loading Llama Vision model for bank statement processing...[/bold yellow]")

# Load model WITHOUT quantization by default
# The bitsandbytes 8-bit quantization can cause RuntimeError during inference
if USE_QUANTIZATION and LOAD_IN_8BIT:
    rprint("[yellow]⚠️ Loading with 8-bit quantization (may cause inference errors)[/yellow]")
    model = MllamaForConditionalGeneration.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.bfloat16,
        device_map=DEVICE_MAP,
        load_in_8bit=True,
    )
else:
    rprint("[green]✅ Loading model without quantization (recommended for H200)[/green]")
    model = MllamaForConditionalGeneration.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.bfloat16,
        device_map=DEVICE_MAP,
    )

processor = AutoProcessor.from_pretrained(MODEL_PATH)

rprint("[bold green]✅ Model loaded successfully[/bold green]")
rprint(f"[cyan]📊 Device: {next(model.parameters()).device}[/cyan]")

if torch.cuda.is_available():
    rprint(f"[magenta]🎮 GPU: {torch.cuda.get_device_name()}[/magenta]")
    rprint(f"[blue]💾 Memory: {torch.cuda.memory_allocated()/1e9:.2f}GB allocated[/blue]")

# Create model info table
model_table = Table(title="🤖 Model Information", border_style="green")
model_table.add_column("Property", style="cyan", no_wrap=True)
model_table.add_column("Value", style="yellow")

model_table.add_row("Model Path", MODEL_PATH)
model_table.add_row("Device", str(next(model.parameters()).device))
model_table.add_row("Quantization", "Disabled" if not USE_QUANTIZATION else "8-bit enabled")
model_table.add_row("Max Tokens", str(MAX_NEW_TOKENS))

if torch.cuda.is_available():
    model_table.add_row("GPU", torch.cuda.get_device_name())
    model_table.add_row("GPU Memory", f"{torch.cuda.memory_allocated()/1e9:.2f}GB")

console.print(model_table)

In [ ]:
# ============================================================================= 
# MINIMAL PROMPT - SEPARATE TABLES PER DATE (BEST WORKING VERSION)
# =============================================================================

MINIMAL_PROMPT = """Extract the transaction table from this bank statement.

# CRITICAL: Create a SEPARATE markdown table for EACH DISTINCT DATE.

IMPORTANT: Look at the ACTUAL COLUMNS in the bank statement image:
- Column 1: Date
- Column 2: Transaction (description)
- Column 3: Debit (amount only appears here if money LEAVES account)
- Column 4: Credit (amount only appears here if money ENTERS account)
- Column 5: Balance (running balance after transaction)

READ THE VISUAL TABLE CAREFULLY:
- If you see an amount in the DEBIT column → put it in Debit, Credit = NOT_FOUND
- If you see an amount in the CREDIT column → put it in Credit, Debit = NOT_FOUND
- ALWAYS copy the balance value exactly as shown

# For each date:
# 1. Create a new markdown table
# 2. If multiple transactions occur on that date, each gets its own row
# 3. Copy the amounts EXACTLY as they appear in the original columns

Example:
### Date: [date]
| Date | Description | Debit | Credit | Balance |
|------|-------------|-------|--------|---------|
| [date] | [description] | [if amount in debit column] | [if amount in credit column] | [balance] |

Use NOT_FOUND only for empty debit/credit cells.
DO NOT classify based on transaction description - use the ACTUAL column position from the image.
"""

print("📋 MINIMAL PROMPT:")
print("="*60)
print(MINIMAL_PROMPT)
print("="*60)

In [ ]:
def extract_bank_statement(
    image_path: str, 
    prompt_type: str = "csv_extraction",
    custom_prompt: str = None,
    temperature: float = 0.1
) -> Dict[str, Any]:
    """
    Extract bank statement data from an image.
    
    Args:
        image_path: Path to bank statement image
        prompt_type: Type of extraction prompt
        custom_prompt: Optional custom prompt
        temperature: Generation temperature (lower = more deterministic)
    
    Returns:
        Dictionary with extraction results
    """
    # Load image
    image = Image.open(image_path)
    
    # Select prompt - default to csv_extraction if not found
    if custom_prompt:
        prompt = custom_prompt
    elif prompt_type == "minimal":
        prompt = MINIMAL_PROMPT
    else:
        prompt = BANK_STATEMENT_PROMPTS.get(
            prompt_type, BANK_STATEMENT_PROMPTS["csv_extraction"]
        )
    
    # Prepare inputs
    messages = [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": prompt}
        ]}
    ]
    
    # Apply chat template
    input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
    
    # Process inputs
    inputs = processor(
        image,
        input_text,
        add_special_tokens=False,
        return_tensors="pt"
    ).to(model.device)
    
    # Generate response with Rich progress indication
    rprint("[yellow]🔍 Analyzing bank statement...[/yellow]")
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=temperature,
            top_p=0.95
        )
    
    # Decode response
    response = processor.decode(output[0], skip_special_tokens=True)
    
    # Extract assistant response
    if "assistant" in response:
        response = response.split("assistant")[-1].strip()
    
    return {
        "raw_response": response,
        "prompt_type": prompt_type,
        "image_path": image_path,
        "extraction_time": datetime.now().isoformat()
    }

# =============================================================================
# QUICK MINIMAL PROMPT TEST - IMMEDIATE FEEDBACK AFTER MODEL LOADING
# =============================================================================

# Load ground truth data to get expected transaction count
def get_expected_transaction_count(image_path: str) -> int:
    """Get the expected number of transactions from ground truth CSV."""
    try:
        # Read ground truth CSV
        ground_truth_df = pd.read_csv(GROUND_TRUTH_CSV)
        
        # Extract just the filename from the path
        from pathlib import Path
        image_filename = Path(image_path).name
        
        # Find the row for this image
        image_row = ground_truth_df[ground_truth_df['image_file'] == image_filename]
        
        if image_row.empty:
            rprint(f"[yellow]⚠️ No ground truth found for {image_filename}[/yellow]")
            return 0
        
        # Count transactions - use TRANSACTION_DATES column
        transaction_dates = image_row.iloc[0]['TRANSACTION_DATES']
        
        if pd.isna(transaction_dates) or transaction_dates == 'NOT_FOUND':
            # Try LINE_ITEM_DESCRIPTIONS as fallback for receipts/invoices
            line_items = image_row.iloc[0]['LINE_ITEM_DESCRIPTIONS']
            if pd.isna(line_items) or line_items == 'NOT_FOUND':
                return 0
            return len(line_items.split(' | '))
        
        # Count the number of dates (transactions)
        return len(transaction_dates.split(' | '))
        
    except Exception as e:
        rprint(f"[red]❌ Error reading ground truth: {e}[/red]")
        return 0

rprint("[bold green]🧪 TESTING WITH MINIMAL PROMPT (Early Position)[/bold green]")
rprint(f"[cyan]📄 Image: {STATEMENT_IMAGE_PATH}[/cyan]")
console.rule("[bold blue]Extraction Test[/bold blue]")

# Test the minimal prompt immediately after model initialization
minimal_result = extract_bank_statement(
    STATEMENT_IMAGE_PATH, 
    custom_prompt=MINIMAL_PROMPT
)

# Display results with Rich formatting
result_panel = Panel(
    minimal_result["raw_response"],
    title="[bold yellow]📊 MINIMAL PROMPT OUTPUT[/bold yellow]",
    border_style="yellow",
    expand=False
)
console.print(result_panel)

# Quick analysis to check for hallucination issues
output_text = minimal_result["raw_response"]
transaction_lines = [line for line in output_text.split('\n') if '|' in line and not 'Date' in line and not '---' in line and line.strip().startswith('|')]

# Get expected count from ground truth
expected_count = get_expected_transaction_count(STATEMENT_IMAGE_PATH)

# Create analysis table
analysis_table = Table(title="📈 Quick Analysis", border_style="green")
analysis_table.add_column("Metric", style="cyan", no_wrap=True)
analysis_table.add_column("Value", style="magenta")
analysis_table.add_column("Status", justify="center")

analysis_table.add_row(
    "Transactions extracted", 
    str(len(transaction_lines)),
    "✅ Correct" if len(transaction_lines) == expected_count else 
    f"⚠️ Mismatch (expected {expected_count})"
)

analysis_table.add_row(
    "Expected from ground truth", 
    str(expected_count),
    "📋 Dynamically loaded from CSV"
)

# Check for obvious hallucination patterns
hallucination_status = "❌ WARNING: Detected hallucination pattern" if ("IGA" in output_text and output_text.count("1102.37") > 5) else "✅ No obvious hallucination detected"
analysis_table.add_row(
    "Hallucination check", 
    "IGA pattern detection",
    hallucination_status
)

console.print(analysis_table)

console.rule("[bold blue]Next Steps[/bold blue]")
rprint("[green]💡 TIP: If results look good, proceed with full prompts below[/green]")
rprint("[red]💡 TIP: If hallucinating, focus on fixing basic vision issues first[/red]")

In [ ]:
# Test with minimal single table prompt
print("🧪 TESTING WITH MINIMAL SINGLE TABLE PROMPT...")
print(f"📄 Image: {STATEMENT_IMAGE_PATH}")
print("="*60)

minimal_result= extract_bank_statement(
    STATEMENT_IMAGE_PATH, 
    custom_prompt=MINIMAL_PROMPT
)

print("\n📊 MINIMAL PROMPT SINGLE TABLE OUTPUT:")
print("-"*60)
print(minimal_result["raw_response"])
print("-"*60)

In [ ]:
# Load and display the bank statement image
print(f"📄 Loading bank statement image: {STATEMENT_IMAGE_PATH}")
try:
    image = Image.open(STATEMENT_IMAGE_PATH)
    print(f"✅ Image loaded successfully: {image.size[0]}x{image.size[1]} pixels")
    
    # Display the image in the notebook
    display(image)
    
except Exception as e:
    print(f"❌ Error loading image: {e}")
    print(f"   Check that the file exists at: {STATEMENT_IMAGE_PATH}")

In [ ]:
# =============================================================================
# LLAMAVISIONTABLEEXTRACTOR CLASS - PRODUCTION-READY IMPLEMENTATION
# =============================================================================

from datetime import datetime
from typing import Dict, List, Any

class LlamaVisionTableExtractor:
    def __init__(self, model_path: str = None, processor=None, model=None):
        """
        Initialize with either a model path OR existing processor/model instances
        """
        if processor is not None and model is not None:
            # Use existing instances from notebook
            self.processor = processor
            self.model = model
        elif model_path:
            # Load new instances (for standalone usage)
            self.processor = AutoProcessor.from_pretrained(model_path)
            self.model = MllamaForConditionalGeneration.from_pretrained(
                model_path,
                torch_dtype=torch.bfloat16,
                device_map="auto"
            )
        else:
            raise ValueError("Must provide either model_path OR (processor, model) instances")
        
    def extract_table(self, image_path: str, prompt_template: str, max_new_tokens: int = 6000) -> Dict[str, Any]:
        """Extract structured data from table image using Llama 3.2 Vision"""
        # Load and preprocess image
        image = Image.open(image_path)
        
        # Apply Llama chat template
        messages = [
            {"role": "user", "content": [
                {"type": "image"},
                {"type": "text", "text": prompt_template}
            ]}
        ]
        
        # Process inputs (Llama 3.2 Vision specific)
        input_text = self.processor.apply_chat_template(messages, add_generation_prompt=True)
        inputs = self.processor(
            image,
            input_text,
            add_special_tokens=False,
            return_tensors="pt"
        ).to(self.model.device)
        
        # Generate extraction with Rich progress indication
        rprint(f"[yellow]🔍 Extracting data from {Path(image_path).name}...[/yellow]")
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=0.0,
                top_p=0.95
            )
        
        # Decode response (extract assistant part)
        response = self.processor.decode(outputs[0], skip_special_tokens=True)
        if "assistant" in response:
            response = response.split("assistant")[-1].strip()
        
        return self._parse_response(response, image_path)
    
    def _parse_response(self, response: str, image_path: str) -> Dict[str, Any]:
        """Parse model response into structured format"""
        return {
            "raw_response": response,
            "image_path": image_path,
            "extraction_time": datetime.now().isoformat(),
            "parsed_data": self._extract_table_data(response),
            "transaction_count": len(self._extract_table_data(response))
        }
    
    def _extract_table_data(self, response: str) -> List[Dict]:
        """Extract table rows from markdown response"""
        lines = response.split('\n')
        table_rows = []
        
        for line in lines:
            if '|' in line and not line.strip().startswith('|---') and 'Date' not in line:
                # Parse table row
                cells = [cell.strip() for cell in line.split('|')[1:-1]]
                if len(cells) >= 4:  # Ensure minimum columns
                    row_data = {
                        'date': cells[0] if cells[0] != 'NOT_FOUND' else None,
                        'description': cells[1] if cells[1] != 'NOT_FOUND' else None,
                        'debit': cells[2] if cells[2] != 'NOT_FOUND' else None,
                        'credit': cells[3] if cells[3] != 'NOT_FOUND' else None,
                        'balance': cells[4] if len(cells) > 4 and cells[4] != 'NOT_FOUND' else None
                    }
                    # Only add if at least one field has data
                    if any(value is not None for value in row_data.values()):
                        table_rows.append(row_data)
        
        return table_rows
    
    def batch_extract(self, image_paths: List[str], prompt_template: str) -> Dict[str, Any]:
        """Extract from multiple images and return consolidated results"""
        results = {}
        total_transactions = 0
        
        for image_path in image_paths:
            try:
                result = self.extract_table(image_path, prompt_template)
                results[image_path] = result
                total_transactions += result['transaction_count']
                rprint(f"[green]✅ {Path(image_path).name}: {result['transaction_count']} transactions[/green]")
            except Exception as e:
                rprint(f"[red]❌ {Path(image_path).name}: Error - {e}[/red]")
                results[image_path] = {"error": str(e)}
        
        return {
            "individual_results": results,
            "summary": {
                "total_images_processed": len(results),
                "total_transactions_extracted": total_transactions,
                "processing_time": datetime.now().isoformat()
            }
        }

rprint("[bold green]✅ LlamaVisionTableExtractor class loaded successfully[/bold green]")

In [ ]:
# =============================================================================
# TEST THE LLAMAVISIONTABLEEXTRACTOR CLASS
# =============================================================================

rprint("[bold cyan]🧪 Testing LlamaVisionTableExtractor with current model...[/bold cyan]")

# Create extractor using existing model and processor from notebook
extractor = LlamaVisionTableExtractor(processor=processor, model=model)

console.rule("[bold blue]Class-based Extraction Test[/bold blue]")

# Test with the current image and minimal prompt
if STATEMENT_IMAGE_PATH:
    result = extractor.extract_table(STATEMENT_IMAGE_PATH, MINIMAL_PROMPT)
    
    # Display results
    rprint(f"[cyan]📄 Image processed:[/cyan] {result['image_path']}")
    rprint(f"[magenta]📊 Transactions found:[/magenta] {result['transaction_count']}")
    rprint(f"[green]⏰ Processing time:[/green] {result['extraction_time']}")
    
    # Show parsed data in a table
    if result['parsed_data']:
        parsed_table = Table(title="📋 Parsed Transaction Data", border_style="green")
        parsed_table.add_column("#", style="dim", width=3)
        parsed_table.add_column("Date", style="cyan")
        parsed_table.add_column("Description", style="white", max_width=30)
        parsed_table.add_column("Debit", style="red")
        parsed_table.add_column("Credit", style="green")
        parsed_table.add_column("Balance", style="yellow")
        
        for i, transaction in enumerate(result['parsed_data'], 1):
            parsed_table.add_row(
                str(i),
                transaction.get('date', ''),
                transaction.get('description', ''),
                transaction.get('debit', ''),
                transaction.get('credit', ''),
                transaction.get('balance', '')
            )
        
        console.print(parsed_table)
    else:
        rprint("[red]❌ No transactions parsed from response[/red]")
    
    # Show raw response
    raw_panel = Panel(
        result["raw_response"][:500] + "..." if len(result["raw_response"]) > 500 else result["raw_response"],
        title="[bold yellow]🔍 Raw Model Response (truncated)[/bold yellow]",
        border_style="yellow",
        expand=False
    )
    console.print(raw_panel)
    
    # Compare with ground truth
    expected_count = get_expected_transaction_count(STATEMENT_IMAGE_PATH)
    
    comparison_table = Table(title="📈 Class vs Function Comparison", border_style="blue")
    comparison_table.add_column("Method", style="cyan")
    comparison_table.add_column("Transactions", style="magenta")
    comparison_table.add_column("Match Ground Truth", style="green")
    
    comparison_table.add_row("Class Method", str(result['transaction_count']), 
                           "✅ Perfect" if result['transaction_count'] == expected_count else f"⚠️ Expected {expected_count}")
    
    console.print(comparison_table)

else:
    rprint("[red]❌ No image path available for testing[/red]")

console.rule("[bold blue]Next Steps[/bold blue]")
rprint("[green]💡 Class loaded successfully - ready for production use![/green]")
rprint("[cyan]💡 Try: extractor.batch_extract([image1, image2], MINIMAL_PROMPT)[/cyan]")